# Online Inference (Standalone)
This notebook is completely standalone. It contains all the required classes and functions directly in the cells. It takes the pre-computed person data from the offline stage and a target clothing image to generate the final try-on result.

> **Note for Kaggle/Colab users**: Please update the checkpoint paths and dataset paths in the configuration section to point to your uploaded datasets.


In [ ]:
import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import cv2

# Configuration - UPDATE THESE PATHS FOR KAGGLE/COLAB
cloth_img_path = "../inputs/test_cloth.jpg"
preprocessed_dir = Path("../outputs/preprocessed")

stableviton_dir = "../StableVITON"
viton_checkpoint = "../checkpoints/stablevton/ckpts/VITONHD.ckpt"
runtime_dir = "../tmp/runtime"

category = "Upper-body" # Upper-body, Lower-body, Dress

print("Setup complete.")


## Utilities


In [ ]:
# ==================================================
# utils/memory.py
# ==================================================

"""
GPU memory management utilities.

Provides stage-based VRAM management to allow sequential loading/unloading
of large models on a single GPU without OOM.

Extracted from the offline_preprocessing notebook's clear_memory() pattern.
"""

import gc
from contextlib import contextmanager

import torch


def clear_memory():
    """Force garbage collection and clear CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


@contextmanager
def gpu_stage(stage_name: str):
    """
    Context manager that clears GPU memory after each pipeline stage.

    Usage:
        with gpu_stage("Person Parsing"):
            model = load_model()
            result = model(input)
            del model
        # VRAM is freed here even if an exception occurs
    """
    print(f"[{stage_name}] Starting...")
    try:
        yield
    finally:
        clear_memory()
        print(f"[{stage_name}] Done. VRAM freed.")


def get_device() -> str:
    """Return 'cuda' if available, else 'cpu'."""
    return "cuda" if torch.cuda.is_available() else "cpu"


def print_vram_usage():
    """Print current VRAM usage (debug helper)."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    else:
        print("  VRAM: N/A (no CUDA)")



In [ ]:
# ==================================================
# utils/image_utils.py
# ==================================================

"""
Image I/O and conversion utilities.

Consolidated from scattered helpers across both notebooks:
- save_rgb, save_mask from offline notebook cell 1
- ensure_rgb, ensure_gray from app_online.py
- resize_rgb from offline notebook cell 1
"""

from pathlib import Path
from typing import Union

import numpy as np
from PIL import Image


# ─── Image Loading ──────────────────────────────────────────────────

def ensure_rgb(path_or_img: Union[Path, str, Image.Image]) -> Image.Image:
    """Open an image and convert to RGB."""
    if isinstance(path_or_img, Image.Image):
        return path_or_img.convert("RGB")
    return Image.open(path_or_img).convert("RGB")


def ensure_gray(path_or_img: Union[Path, str, Image.Image]) -> Image.Image:
    """Open an image and convert to grayscale."""
    if isinstance(path_or_img, Image.Image):
        return path_or_img.convert("L")
    return Image.open(path_or_img).convert("L")


# ─── Image Saving ───────────────────────────────────────────────────

def save_rgb(img: Image.Image, path: Union[Path, str]):
    """Save an RGB image, creating parent directories if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)


def save_mask(mask: Union[Image.Image, np.ndarray], path: Union[Path, str]):
    """
    Save a mask image (grayscale), handling various input formats:
    - PIL Image → save directly as "L"
    - bool ndarray → convert to 0/255
    - float ndarray with max <= 1 → scale to 0/255
    - uint8 ndarray → save directly
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(mask, Image.Image):
        mask.convert("L").save(path)
        return

    mask = np.array(mask)
    if mask.dtype == bool:
        mask = mask.astype(np.uint8) * 255
    elif mask.max() <= 1:
        mask = mask.astype(np.uint8) * 255
    else:
        mask = mask.astype(np.uint8)

    Image.fromarray(mask).save(path)


# ─── Image Resizing ─────────────────────────────────────────────────

def resize_rgb(img: Image.Image, size: tuple = (768, 1024)) -> Image.Image:
    """Resize an RGB image to (width, height) using bicubic interpolation."""
    return img.resize(size, Image.BICUBIC)


# ─── Validation ──────────────────────────────────────────────────────

def validate_nonempty_mask(
    mask_path: Union[Path, str], min_positive_pixels: int = 100
) -> tuple:
    """
    Check that a mask file exists and has sufficient positive (white) area.

    Returns:
        (ok: bool, message: str)
    """
    mask_path = Path(mask_path)
    if not mask_path.exists():
        return False, "file_not_found"

    arr = np.array(Image.open(mask_path).convert("L"))
    positive = int((arr > 127).sum())
    if positive < min_positive_pixels:
        return False, f"too_small_positive_area={positive}"
    return True, f"positive_area={positive}"



In [ ]:
# ==================================================
# postprocess/mask_utils.py
# ==================================================

"""
Mask morphology and selection utilities.

Extracted from offline_preprocessing notebook cell 2.
Provides mask cleaning, label selection, connected component filtering,
and agnostic image/mask generation for VITON preprocessing.
"""

import cv2
import numpy as np
from PIL import Image


# ─── Morphological Operations ───────────────────────────────────────

def clean_mask(mask: np.ndarray, open_k: int = 5, close_k: int = 7, blur_k: int = 0) -> np.ndarray:
    """
    Clean a binary mask using morphological open/close and optional blur.

    Args:
        mask: Binary mask (0/255 uint8 or bool).
        open_k: Kernel size for morphological opening (removes small noise).
        close_k: Kernel size for morphological closing (fills small holes).
        blur_k: Kernel size for Gaussian blur (0 = disabled).

    Returns:
        Cleaned binary mask (0/255 uint8).
    """
    mask = mask.astype(np.uint8)

    if open_k > 0:
        kernel_open = np.ones((open_k, open_k), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    if close_k > 0:
        kernel_close = np.ones((close_k, close_k), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    if blur_k and blur_k > 1:
        if blur_k % 2 == 0:
            blur_k += 1
        mask = cv2.GaussianBlur(mask, (blur_k, blur_k), 0)

    mask = (mask > 127).astype(np.uint8) * 255
    return mask


def keep_largest_component(mask: np.ndarray) -> np.ndarray:
    """
    Keep only the largest connected component in a binary mask.

    Args:
        mask: Binary mask (0/255 uint8).

    Returns:
        Mask with only the largest connected component.
    """
    mask_bin = (mask > 127).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)

    if num_labels <= 1:
        return (mask_bin * 255).astype(np.uint8)

    largest_idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    largest = (labels == largest_idx).astype(np.uint8) * 255
    return largest


def refine_top_mask(top_mask: np.ndarray, open_k: int = 5, close_k: int = 9) -> np.ndarray:
    """
    Refine a clothing/top mask: clean (Removed keep_largest_component to avoid dropping disconnected sleeves).
    """
    top_mask = clean_mask(top_mask, open_k=open_k, close_k=close_k)
    return top_mask


# ─── Label Selection ────────────────────────────────────────────────

def get_binary_mask_from_labels(label_map: np.ndarray, target_labels: list) -> np.ndarray:
    """Create a binary mask (0/255) from selected labels in a segmentation map."""
    mask = np.isin(label_map, target_labels).astype(np.uint8) * 255
    return mask


def pick_best_top_labels(
    label_map: np.ndarray,
    candidate_label_sets: list,
    min_area_ratio: float = 0.01,
    max_area_ratio: float = 0.55,
) -> tuple:
    """
    Choose the best set of labels for the clothing region based on mask area.

    Iterates through candidate label sets, computes the mask area for each,
    and picks the one with the largest area within acceptable bounds.

    Args:
        label_map: Segmentation map (H, W) with integer labels.
        candidate_label_sets: List of label sets to try, e.g. [[4], [5], [4,5]].
        min_area_ratio: Minimum acceptable mask area / total area.
        max_area_ratio: Maximum acceptable mask area / total area.

    Returns:
        (best_labels: list, best_mask: np.ndarray)
    """
    H, W = label_map.shape
    total_area = H * W

    best_labels = None
    best_mask = None
    best_score = -1

    for labels in candidate_label_sets:
        mask = get_binary_mask_from_labels(label_map, labels)
        mask = clean_mask(mask, open_k=3, close_k=7)
        area_ratio = (mask > 127).sum() / total_area

        if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
            continue

        score = area_ratio
        if score > best_score:
            best_score = score
            best_labels = labels
            best_mask = mask

    if best_labels is None:
        # Fallback to label [4] if no candidate is suitable
        best_labels = [4]
        best_mask = get_binary_mask_from_labels(label_map, best_labels)

    return best_labels, best_mask


# ─── Agnostic Generation ────────────────────────────────────────────

def create_robust_agnostic(
    person_img_pil: Image.Image,
    parsing_map: np.ndarray,
    top_labels: list = None,
    object_mask: np.ndarray = None,
    category: str = "Upper-body",
) -> tuple:
    """
    Create agnostic image and mask for VITON inference.

    Erases the clothing region (+ optionally occluding objects like bags)
    and exposed skin (arms, neck) from the person image, replacing
    them with a neutral gray (127).

    Args:
        person_img_pil: Original person image (RGB PIL).
        parsing_map: Segmentation map from SegFormer (H, W).
        top_labels: Label IDs for the upper-body clothing region.
        object_mask: Optional binary mask (0/255) for occluding objects.
        category: Garment category ("Upper-body", "Lower-body", "Dress").

    Returns:
        (agnostic_img: PIL.Image, agnostic_mask: PIL.Image)
    """
    if top_labels is None:
        top_labels = [4]

    person_np = np.array(person_img_pil)

    # 1. Get clothing mask
    mask_cloth = np.isin(parsing_map, top_labels).astype(np.uint8) * 255

    # 2. Merge object mask (e.g. bag) if provided
    if object_mask is not None:
        mask_object = (object_mask > 127).astype(np.uint8) * 255
        mask_cloth = np.maximum(mask_cloth, mask_object)

    # 3. Get skin mask
    # SegFormer mattmdjaga/segformer_b2_clothes labels:
    # 12 = left-leg, 13 = right-leg, 14 = right-arm, 15 = left-arm
    skin_labels = [14, 15]
    if category == "Lower-body":
        skin_labels = [12, 13]
    elif category == "Dress":
        skin_labels = [12, 13, 14, 15]
        
    mask_skin = np.isin(parsing_map, skin_labels).astype(np.uint8) * 255

    # 4. Combine clothing + skin → dilate to cover edges
    combined_mask = np.maximum(mask_cloth, mask_skin)
    kernel = np.ones((15, 15), np.uint8)
    agnostic_mask_np = cv2.dilate(combined_mask, kernel, iterations=2)

    # Prevent lower-body mask from bleeding up into the upper clothes
    if category == "Lower-body":
        mask_preserve = np.isin(parsing_map, [4]).astype(np.uint8) * 255
        agnostic_mask_np[mask_preserve > 127] = 0

    # 5. PROTECT REGIONS FROM BEING ERASED/DRAWN OVER
    # Protect head/face/hair (labels: 1=Hat, 2=Hair, 3=Sunglasses, 11=Face)
    mask_head = np.isin(parsing_map, [1, 2, 3, 11]).astype(np.uint8) * 255
    agnostic_mask_np[mask_head > 127] = 0

    # 6. Create agnostic image: fill masked regions with neutral gray
    agnostic_np = person_np.copy()
    agnostic_np[agnostic_mask_np > 127] = 127

    return Image.fromarray(agnostic_np), Image.fromarray(agnostic_mask_np)


# ─── Feather Mask (for blending) ────────────────────────────────────

def feather_mask(mask_pil: Image.Image, ksize: int = 15) -> np.ndarray:
    """
    Create a soft-edged alpha mask for blending.

    Args:
        mask_pil: Binary mask (PIL Image).
        ksize: Gaussian blur kernel size for feathering.

    Returns:
        Float alpha mask (H, W, 1) in range [0, 1].
    """
    mask = np.array(mask_pil.convert("L")).astype(np.uint8)
    mask = (mask > 127).astype(np.uint8) * 255

    if ksize % 2 == 0:
        ksize += 1

    mask = cv2.GaussianBlur(mask, (ksize, ksize), 0)
    return (mask.astype(np.float32) / 255.0)[..., None]



## Stage 5: Cloth Parsing


In [ ]:
# ==================================================
# stages/cloth_parsing.py
# ==================================================

"""
Stage 5: Cloth Parsing using SegFormer.

Segments the cloth image to create a binary mask of the clothing region.
Uses the same SegFormer model as person parsing but with different
label selection logic optimized for isolated clothing images.

Extracted from offline_preprocessing notebook cells 22-23.
Model: mattmdjaga/segformer_b2_clothes (~1GB VRAM, shared with PersonParsing)
"""

from typing import Optional

import numpy as np
import torch
from PIL import Image



CLOTH_CATEGORY_CANDIDATES = {
    "Upper-body": [[4], [4, 7], [7], [4, 5], [4, 6]],
    "Lower-body": [[6], [5], [5, 6]],
    "Dress": [[7], [4, 7], [4, 5, 6]],
}

PARSING_MODEL_ID = "mattmdjaga/segformer_b2_clothes"


class ClothParsingStage:
    """SegFormer-based cloth mask generation."""

    def __init__(self, device: Optional[str] = None):
        self.device = device or get_device()
        self.processor = None
        self.model = None

    def load(self):
        """Load SegFormer model into VRAM."""
        from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

        self.processor = AutoImageProcessor.from_pretrained(PARSING_MODEL_ID)
        self.model = AutoModelForSemanticSegmentation.from_pretrained(
            PARSING_MODEL_ID
        ).to(self.device)
        self.model.eval()
        print(f"[ClothParsing] SegFormer loaded on {self.device}")

    def unload(self):
        """Free model from VRAM."""
        del self.model, self.processor
        self.model = self.processor = None
        clear_memory()

    @torch.no_grad()
    def _run_segformer(self, image_pil: Image.Image) -> np.ndarray:
        """Run SegFormer inference and return the label map."""
        inputs = self.processor(images=image_pil, return_tensors="pt").to(self.device)
        outputs = self.model(**inputs)
        logits = outputs.logits

        upsampled = torch.nn.functional.interpolate(
            logits,
            size=image_pil.size[::-1],  # (H, W)
            mode="bilinear",
            align_corners=False,
        )
        pred = upsampled.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        return pred

    def run(
        self,
        cloth_img: Image.Image,
        person_top_labels: Optional[list] = None,
        category: str = "Upper-body",
    ) -> dict:
        """
        Parse cloth image and generate binary cloth mask.

        Args:
            cloth_img: Cloth image (PIL RGB).
            person_top_labels: If available, prioritize matching labels
                              from person parsing for consistency.
            category: Garment category ("Upper-body", "Lower-body", "Dress").

        Returns:
            dict with keys:
                - "cloth_mask": np.ndarray (0/255) binary cloth mask
                - "cloth_parsing_map": np.ndarray segmentation map
                - "cloth_labels": list of selected label IDs
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        # 1. Run segmentation
        cloth_parsing_map = self._run_segformer(cloth_img)

        # 2. Build candidate list: prioritize person's labels if available
        candidates = list(CLOTH_CATEGORY_CANDIDATES[category])
        if person_top_labels:
            if person_top_labels not in candidates:
                candidates.insert(0, person_top_labels)
            else:
                candidates.remove(person_top_labels)
                candidates.insert(0, person_top_labels)

        # 3. Pick best labels
        cloth_labels, cloth_mask = pick_best_top_labels(
            cloth_parsing_map, candidates,
            min_area_ratio=0.05, max_area_ratio=0.90,
        )

        # 4. Refine mask
        cloth_mask = refine_top_mask(cloth_mask, open_k=5, close_k=9)

        print(f"[ClothParsing] Labels selected: {cloth_labels}, "
              f"mask area: {(cloth_mask > 127).sum()}")

        return {
            "cloth_mask": cloth_mask,
            "cloth_parsing_map": cloth_parsing_map,
            "cloth_labels": cloth_labels,
        }



## Stage 6: Try-On Inference


In [ ]:
# ==================================================
# stages/tryon_inference.py
# ==================================================

"""
Stage 6: StableVITON Try-On Inference.

Wraps the StableVITON inference.py CLI to run virtual try-on. This stage:
1. Prepares the VITON-HD test data directory structure
2. Invokes StableVITON's inference.py as a subprocess
3. Collects the output image

This is the heaviest stage (~10-12GB VRAM). It is always run last so
that VRAM cleanup is unnecessary.

Extracted from online_inference notebook (patched V10 version).
Model: StableVITON VITONHD checkpoint
"""

import shutil
import subprocess
import sys
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
from PIL import Image


class TryOnInferenceStage:
    """StableVITON inference wrapper using subprocess CLI."""

    def __init__(
        self,
        stableviton_dir: Path,
        checkpoint_path: Path,
        runtime_dir: Path,
    ):
        """
        Args:
            stableviton_dir: Path to the StableVITON repo root.
            checkpoint_path: Path to VITONHD.ckpt.
            runtime_dir: Working directory for temporary data and outputs.
        """
        self.stableviton_dir = Path(stableviton_dir)
        self.checkpoint_path = Path(checkpoint_path)
        self.runtime_dir = Path(runtime_dir)

        self.config_path = self.stableviton_dir / "configs" / "VITONHD.yaml"
        self.data_root = self.runtime_dir / "data" / "viton_hd"
        self.output_root = self.runtime_dir / "outputs"

    def _prepare_test_sample(
        self,
        person_img: Image.Image,
        cloth_img: Image.Image,
        cloth_mask: Image.Image,
        agnostic_img: Image.Image,
        agnostic_mask: Image.Image,
        densepose_img: Image.Image,
        object_mask: Optional[np.ndarray] = None,
        sample_name: str = "00000_00.jpg",
    ):
        """
        Prepare the VITON-HD test directory structure with a single sample.

        This implements the V10 patched version from the online notebook:
        - Merges object mask (bag) into agnostic mask
        - Dilates combined mask to cover edges
        - Creates clean gray-filled agnostic image

        The directory structure created is:
            data/viton_hd/test/
                image/          → person image
                cloth/          → clothing image
                cloth-mask/     → clothing binary mask
                agnostic-v3.2/  → agnostic (clothing-erased) image
                agnostic-mask/  → agnostic binary mask
                image-densepose/ → body pose map
            data/viton_hd/test_pairs.txt
        """
        test_root = self.data_root / "test"

        folders = [
            "image", "cloth", "cloth-mask",
            "agnostic-mask", "agnostic-v3.2", "image-densepose",
        ]

        # Clean previous run
        if self.data_root.exists():
            shutil.rmtree(self.data_root)

        for folder in folders:
            (test_root / folder).mkdir(parents=True, exist_ok=True)

        # V10 patch: merge object mask into agnostic if provided
        if object_mask is not None:
            agnostic_mask_np = np.array(agnostic_mask.convert("L")) > 127
            object_mask_bool = object_mask > 127

            combined_mask_np = np.maximum(
                agnostic_mask_np.astype(np.uint8),
                object_mask_bool.astype(np.uint8),
            )

            # NOTE: Removed 25x25 dilation here because mask_utils.py already dilates by 30px!
            # Adding another 25px dilation causes the mask to bleed severely.
            agnostic_mask = Image.fromarray((combined_mask_np * 255).astype(np.uint8))

            # Rebuild agnostic image with combined mask
            person_np = np.array(person_img).copy()
            person_np[combined_mask_np > 0] = 127
            agnostic_img = Image.fromarray(person_np)

        mask_name = sample_name.replace(".jpg", "_mask.png")

        # Save all inputs
        person_img.save(test_root / "image" / sample_name)
        cloth_img.save(test_root / "cloth" / sample_name)
        cloth_mask.save(test_root / "cloth-mask" / sample_name)
        agnostic_img.save(test_root / "agnostic-v3.2" / sample_name)
        densepose_img.save(test_root / "image-densepose" / sample_name)

        # Save agnostic mask in both formats (some code paths expect each)
        agnostic_mask.save(test_root / "agnostic-mask" / mask_name)
        agnostic_mask.save(test_root / "agnostic-mask" / sample_name.replace(".jpg", ".png"))

        # Write test pairs file
        with open(self.data_root / "test_pairs.txt", "w") as f:
            f.write(f"{sample_name} {sample_name}\n")

        print(f"[TryOnInference] Test sample prepared at {self.data_root}")

    def _run_subprocess(self):
        """Run StableVITON inference.py as a subprocess."""
        if self.output_root.exists():
            shutil.rmtree(self.output_root)
        self.output_root.mkdir(parents=True, exist_ok=True)

        python_exe = sys.executable

        cmd = [
            python_exe,
            "inference.py",
            "--config_path", str(self.config_path),
            "--batch_size", "1",
            "--model_load_path", str(self.checkpoint_path),
            "--save_dir", str(self.output_root),
            "--data_root_dir", str(self.data_root),
        ]

        print(f"[TryOnInference] Running: {' '.join(cmd)}")
        print(f"[TryOnInference] CWD: {self.stableviton_dir}")

        result = subprocess.run(
            cmd,
            cwd=str(self.stableviton_dir),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )

        if result.returncode != 0:
            print("[TryOnInference] STDOUT:", result.stdout)
            print("[TryOnInference] STDERR:", result.stderr)
            raise RuntimeError(f"StableVITON inference failed (exit code {result.returncode})")

        print("[TryOnInference] Inference completed successfully.")

    def _find_latest_result(self) -> Path:
        """Find the most recently written output image."""
        candidates = list(self.output_root.rglob("*.png")) + list(self.output_root.rglob("*.jpg"))
        if not candidates:
            raise FileNotFoundError(f"No output found in {self.output_root}")
        return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]

    def run(
        self,
        person_img: Image.Image,
        cloth_img: Image.Image,
        cloth_mask: Image.Image,
        agnostic_img: Image.Image,
        agnostic_mask: Image.Image,
        densepose_img: Image.Image,
        object_mask: Optional[np.ndarray] = None,
    ) -> Image.Image:
        """
        Run the full StableVITON inference pipeline.

        Args:
            person_img: Original person image (PIL RGB).
            cloth_img: Target clothing image (PIL RGB).
            cloth_mask: Binary mask of the clothing (PIL L).
            agnostic_img: Agnostic person image (PIL RGB).
            agnostic_mask: Agnostic mask (PIL L).
            densepose_img: DensePose visualization (PIL RGB).
            object_mask: Optional object mask for V10 patching (np.ndarray 0/255).

        Returns:
            Try-on result image (PIL RGB).
        """
        self._prepare_test_sample(
            person_img, cloth_img, cloth_mask,
            agnostic_img, agnostic_mask, densepose_img,
            object_mask=object_mask,
        )

        self._run_subprocess()

        result_path = self._find_latest_result()
        result_img = Image.open(result_path).convert("RGB")

        print(f"[TryOnInference] Result: {result_path} ({result_img.size})")
        return result_img



## Stage 7: Post-processing (Alpha Blending)


In [ ]:
# ==================================================
# postprocess/occluder.py
# ==================================================

"""
Occluder restoration (bag/accessory preservation).

Implements the V10 refined alpha-blending approach from the online_inference
notebook. After StableVITON generates the try-on result, objects like bags
that were occluding the original clothing need to be composited back onto
the result image.
"""

import cv2
import numpy as np
from PIL import Image


def refined_restore_occluder(
    original_img: Image.Image,
    result_img: Image.Image,
    object_mask: Image.Image,
    erode_iter: int = 1,
    blur_ksize: int = 5,
) -> Image.Image:
    """
    Restore an occluding object (e.g. bag) from the original image onto
    the try-on result using refined alpha blending.

    The mask is first eroded to trim artifact edges, then blurred for
    smooth blending. Both the original image and mask are resized to
    match the result image dimensions.

    Args:
        original_img: Original person image (PIL RGB).
        result_img: StableVITON output image (PIL RGB).
        object_mask: Binary mask of the object to restore (PIL L).
        erode_iter: Number of erosion iterations (higher = more edge trimming).
        blur_ksize: Gaussian blur kernel size for mask feathering.

    Returns:
        Blended result image with the occluder restored (PIL RGB).
    """
    target_size = result_img.size  # (width, height)

    # Resize original and mask to match result dimensions
    orig_resized = original_img.resize(target_size, Image.Resampling.LANCZOS)
    mask_resized = object_mask.resize(target_size, Image.Resampling.NEAREST)

    orig_np = np.array(orig_resized).astype(np.float32)
    res_np = np.array(result_img).astype(np.float32)
    mask_np = np.array(mask_resized.convert("L"))

    # 1. Erode mask to trim artifact edges
    kernel_erode = np.ones((3, 3), np.uint8)
    eroded_mask = cv2.erode(mask_np, kernel_erode, iterations=erode_iter)

    # 2. Blur mask edges for smooth blending
    if blur_ksize > 0:
        if blur_ksize % 2 == 0:
            blur_ksize += 1
        blurred_mask = cv2.GaussianBlur(eroded_mask, (blur_ksize, blur_ksize), 0)
    else:
        blurred_mask = eroded_mask

    # 3. Alpha blending: original * alpha + result * (1 - alpha)
    alpha = blurred_mask.astype(np.float32) / 255.0
    alpha = alpha[..., np.newaxis]

    blended_np = orig_np * alpha + res_np * (1.0 - alpha)

    return Image.fromarray(blended_np.astype(np.uint8))



## Execution
Load data and run the inference pipeline.


In [ ]:

cloth_img = Image.open(cloth_img_path).convert("RGB")
IMG_SIZE = (768, 1024)
cloth_img = resize_rgb(cloth_img, IMG_SIZE)
display(cloth_img.resize((384, 512)))

person_img = Image.open(preprocessed_dir / "person_img.png").convert("RGB")
agnostic_img = Image.open(preprocessed_dir / "agnostic_img.png").convert("RGB")
agnostic_mask = Image.open(preprocessed_dir / "agnostic_mask.png").convert("L")
densepose_img = Image.open(preprocessed_dir / "densepose_img.png").convert("RGB")
top_labels = np.load(preprocessed_dir / "top_labels.npy").tolist()

restore_mask_path = preprocessed_dir / "final_restore_mask.png"
final_restore_mask_pil = Image.open(restore_mask_path).convert("L") if restore_mask_path.exists() else None

with gpu_stage("Stage 5: Cloth Parsing"):
    cloth_parser = ClothParsingStage()
    cloth_parser.load()
    cloth_results = cloth_parser.run(cloth_img, person_top_labels=top_labels, category=category)
    cloth_parser.unload()

cloth_mask_np = cloth_results["cloth_mask"]
cloth_mask_pil = Image.fromarray(cloth_mask_np).convert("L")
display(cloth_mask_pil.resize((384, 512)))

# Stage 6: Try-On Inference
tryon = TryOnInferenceStage(
    stableviton_dir=Path(stableviton_dir),
    checkpoint_path=Path(viton_checkpoint),
    runtime_dir=Path(runtime_dir)
)

result_img = tryon.run(
    person_img=person_img,
    cloth_img=cloth_img,
    cloth_mask=cloth_mask_pil,
    agnostic_img=agnostic_img,
    agnostic_mask=agnostic_mask,
    densepose_img=densepose_img,
    object_mask=None 
)

display(result_img.resize((384, 512)))

# Stage 7: Post-processing
if final_restore_mask_pil is not None:
    result_img = refined_restore_occluder(person_img, result_img, final_restore_mask_pil, erode_iter=1, blur_ksize=5)

display(result_img.resize((384, 512)))
result_img.save("../outputs/final_tryon_result.png")
print("Try-on complete! Result saved.")

